# v2 Leakage-Aware Kidney CT Benchmark

This notebook tests a stronger novelty claim than another architecture result:
random image-level splitting on the public CT kidney JPEG dataset can leak exact
duplicates, perceptual near-duplicates, and neighboring slices across
train/validation/test. It builds reproducible leakage-aware splits, trains the
same EfficientNet-B4 baseline across split modes, and reports accuracy, macro
F1, calibration, and Grad-CAM sanity checks.

**Important:** the public JPEG files do not provide reliable patient/study IDs.
The `sequence_proxy` split is a slice/case proxy, not a patient-disjoint split.



## 1) Setup and Configuration

Default `execution_mode="smoke"` trains only the random split for one epoch.
Change it to `"full"` for the complete four-split benchmark.



In [ ]:
import math, os, re, time, random, hashlib, warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import Counter
from itertools import combinations
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

try:
    from sklearn.metrics import roc_auc_score
except Exception:
    roc_auc_score = None

warnings.filterwarnings("ignore", category=UserWarning)

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"]=str(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
torch.backends.cudnn.deterministic=True
torch.backends.cudnn.benchmark=False
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "| PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))



In [ ]:
@dataclass
class BenchmarkConfig:
    execution_mode: str = "smoke"  # "audit_only", "smoke", or "full"
    output_dir: Optional[str] = None
    preferred_data_root: str = (
        "/kaggle/input/datasets/nazmul0087/ct-kidney-dataset-normal-cyst-tumor-and-stone/"
        "CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone/CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone"
    )
    local_data_root: str = "CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone"
    train_split: float = 0.70
    val_split: float = 0.15
    test_split: float = 0.15
    split_seed: int = 42
    near_duplicate_hamming: int = 4
    sequence_gap: int = 5
    # Coarse contiguous filename block size for sequence_proxy.
    # The public JPEG names are consecutive, so gap-only grouping collapses each class.
    sequence_block_size: int = 64
    num_classes: int = 4
    input_resolution: int = 380
    batch_size: int = 16
    full_epochs: int = 20
    smoke_epochs: int = 1
    early_stop_patience: int = 5
    learning_rate: float = 3e-4
    weight_decay: float = 1e-4
    # Kaggle notebooks can throw multiprocessing cleanup assertions with worker processes.
    # Keep this at 0 for reliable full runs; raise only after the notebook is stable.
    num_workers: int = 0
    amp: bool = True
    label_smoothing: float = 0.1
    grad_clip_norm: float = 1.0
    width_coefficient: float = 1.4
    depth_coefficient: float = 1.8
    dropout_rate: float = 0.4
    drop_connect_rate: float = 0.2
    depth_divisor: int = 8
    attention_type: str = "se"
    mean: Tuple[float,float,float] = (0.485,0.456,0.406)
    std: Tuple[float,float,float] = (0.229,0.224,0.225)

CFG=BenchmarkConfig()
if CFG.output_dir is None:
    CFG.output_dir = "/kaggle/working/leakage_aware_benchmark" if Path("/kaggle/working").exists() else "leakage_aware_benchmark_outputs"
OUTPUT_DIR=Path(CFG.output_dir); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(CFG); print("Outputs:", OUTPUT_DIR)



## 2) Dataset Discovery and Image Index



In [ ]:
IMAGE_EXTS={".png",".jpg",".jpeg",".bmp",".tif",".tiff",".webp"}
EXPECTED_CLASSES={"Cyst","Normal","Stone","Tumor"}

def looks_like_imagefolder(root: Path) -> bool:
    if not root.exists() or not root.is_dir(): return False
    dirs=[p for p in root.iterdir() if p.is_dir()]
    if not EXPECTED_CLASSES.intersection({p.name for p in dirs}): return False
    return any(f.is_file() and f.suffix.lower() in IMAGE_EXTS for d in dirs for f in d.rglob("*"))

def find_imagefolder_root(path: Path, max_depth=6):
    if looks_like_imagefolder(path): return path
    frontier=[path]
    for _ in range(max_depth):
        nxt=[]
        for p in frontier:
            children=[c for c in p.iterdir() if c.is_dir()] if p.exists() else []
            for c in children:
                if looks_like_imagefolder(c): return c
            nxt.extend(children)
        frontier=nxt
    return None

def resolve_dataset_root(cfg):
    candidates=[Path(cfg.preferred_data_root), Path(cfg.local_data_root)]
    if Path("/kaggle/input").exists(): candidates += list(Path("/kaggle/input").iterdir())
    for c in candidates:
        found=find_imagefolder_root(c) if c.exists() else None
        if found is not None: return found
    raise FileNotFoundError("Attach the Kaggle dataset or set CFG.preferred_data_root.")

def extract_numeric_index(name):
    m=re.search(r"\((\d+)\)", name)
    if m: return int(m.group(1))
    nums=re.findall(r"\d+", name)
    return int(nums[-1]) if nums else None

def build_image_index(root):
    class_dirs=sorted([p for p in root.iterdir() if p.is_dir() and p.name in EXPECTED_CLASSES], key=lambda p:p.name)
    class_to_idx={p.name:i for i,p in enumerate(class_dirs)}
    rows=[]
    for d in class_dirs:
        for p in sorted(d.rglob("*")):
            if not p.is_file() or p.suffix.lower() not in IMAGE_EXTS: continue
            with Image.open(p) as im:
                w,h=im.size; mode=im.mode
            rows.append(dict(path=str(p.resolve()), relative_path=p.relative_to(root).as_posix(),
                label=class_to_idx[d.name], class_name=d.name, filename=p.name, stem=p.stem,
                numeric_index=extract_numeric_index(p.name), width=w, height=h, mode=mode, file_size=p.stat().st_size))
    return pd.DataFrame(rows).sort_values(["class_name","numeric_index","filename"], na_position="last").reset_index(drop=True)

DATA_ROOT=resolve_dataset_root(CFG)
index_df=build_image_index(DATA_ROOT)
CLASS_NAMES=[x[0] for x in sorted(index_df[["class_name","label"]].drop_duplicates().itertuples(index=False), key=lambda x:x[1])]
print("Dataset root:", DATA_ROOT); print("Images:", len(index_df)); print("Classes:", CLASS_NAMES)
display(index_df.groupby("class_name").size().rename("count").to_frame())
display(index_df.head())



## 3) Exact and Perceptual Leakage Audit



In [ ]:
def sha256_file(path, chunk_size=1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(chunk_size), b""): h.update(chunk)
    return h.hexdigest()

def average_hash(path, size=16):
    with Image.open(path) as im:
        arr=np.asarray(im.convert("L").resize((size,size), Image.Resampling.LANCZOS), dtype=np.float32)
    v=0
    for b in (arr >= arr.mean()).ravel(): v=(v<<1)|int(b)
    return int(v)

def difference_hash(path, size=16):
    with Image.open(path) as im:
        arr=np.asarray(im.convert("L").resize((size+1,size), Image.Resampling.LANCZOS), dtype=np.float32)
    v=0
    for b in (arr[:,1:] >= arr[:,:-1]).ravel(): v=(v<<1)|int(b)
    return int(v)

hash_df=index_df.copy()
hash_cache=OUTPUT_DIR/"image_hash_index.csv"
if hash_cache.exists():
    cached=pd.read_csv(hash_cache)
    if set(cached.relative_path)==set(hash_df.relative_path):
        hash_df=hash_df.merge(cached[["relative_path","sha256","ahash_int","dhash_int","ahash_hex","dhash_hex"]], on="relative_path", how="left")
        print("Loaded cached hashes:", hash_cache)
if "sha256" not in hash_df or hash_df.sha256.isna().any():
    tqdm.pandas(desc="SHA-256"); hash_df["sha256"]=hash_df.path.progress_apply(sha256_file)
    tqdm.pandas(desc="Average hash"); hash_df["ahash_int"]=hash_df.path.progress_apply(average_hash)
    tqdm.pandas(desc="Difference hash"); hash_df["dhash_int"]=hash_df.path.progress_apply(difference_hash)
    hash_df["ahash_hex"]=hash_df.ahash_int.apply(lambda x:f"{int(x):064x}")
    hash_df["dhash_hex"]=hash_df.dhash_int.apply(lambda x:f"{int(x):064x}")
    hash_df.to_csv(hash_cache, index=False)
    print("Saved hash cache:", hash_cache)

exact_groups=hash_df.groupby("sha256").agg(n=("relative_path","size"), classes=("class_name",lambda s:", ".join(sorted(set(s)))), examples=("relative_path",lambda s:" | ".join(list(s)[:5]))).reset_index().sort_values("n", ascending=False)
exact_dupes=exact_groups[exact_groups.n>1]
cross_class_exact=exact_dupes[exact_dupes.classes.str.contains(",")]
print("Exact duplicate groups:", len(exact_dupes))
print("Cross-class exact duplicate groups:", len(cross_class_exact))
display(exact_dupes.head(20))
if len(cross_class_exact): display(cross_class_exact.head(20))



In [ ]:
def hamming_int(a,b): return int((int(a)^int(b)).bit_count())

class UnionFind:
    def __init__(self, items):
        self.parent={int(x):int(x) for x in items}; self.rank={int(x):0 for x in items}
    def find(self,x):
        x=int(x)
        if self.parent[x]!=x: self.parent[x]=self.find(self.parent[x])
        return self.parent[x]
    def union(self,a,b):
        ra,rb=self.find(a),self.find(b)
        if ra==rb: return
        if self.rank[ra]<self.rank[rb]: ra,rb=rb,ra
        self.parent[rb]=ra
        if self.rank[ra]==self.rank[rb]: self.rank[ra]+=1

class BKTree:
    def __init__(self, dist): self.dist=dist; self.root=None
    def add(self, value, item):
        node={"value":int(value),"items":[int(item)],"children":{}}
        if self.root is None: self.root=node; return
        cur=self.root
        while True:
            d=self.dist(value,cur["value"])
            if d==0: cur["items"].append(int(item)); return
            if d not in cur["children"]: cur["children"][d]=node; return
            cur=cur["children"][d]
    def query(self, value, threshold):
        out=[]; stack=[] if self.root is None else [self.root]
        while stack:
            cur=stack.pop(); d=self.dist(value,cur["value"])
            if d<=threshold: out.extend(cur["items"])
            for edge,child in cur["children"].items():
                if d-threshold <= edge <= d+threshold: stack.append(child)
        return out

def build_perceptual_components(df, threshold):
    uf=UnionFind(df.index)
    for _,part in tqdm(df.groupby("class_name"), desc="Perceptual components"):
        for col in ["ahash_int","dhash_int"]:
            tree=BKTree(hamming_int)
            for idx,value in part[col].items():
                for j in tree.query(int(value), threshold): uf.union(idx,j)
                tree.add(int(value), idx)
    roots=[uf.find(i) for i in df.index]
    root_to_id={r:f"pdup_{n:05d}" for n,r in enumerate(sorted(set(roots)))}
    return pd.Series([root_to_id[r] for r in roots], index=df.index, name="phash_group")

def build_sequence_proxy_groups(df, max_gap, block_size=64):
    """Build coarse contiguous filename blocks as a slice/case proxy.

    A pure gap-connected component rule is unusable for this dataset because the
    filenames are almost perfectly consecutive within each class, which collapses
    each class into one giant component. These fixed-size contiguous blocks keep
    neighboring slices mostly together while still allowing train/val/test splits.
    This is not patient-disjoint and should be described only as a proxy split.
    """
    groups=pd.Series(index=df.index, dtype="object", name="sequence_group"); k=0
    for cls,part in df.groupby("class_name"):
        cur=None; last=None; count=0
        numbered=part.dropna(subset=["numeric_index"]).sort_values("numeric_index")
        for idx,row in numbered.iterrows():
            num=int(row.numeric_index)
            start_new = cur is None or last is None or (num-last)>max_gap or count>=block_size
            if start_new:
                cur=f"seq_{cls}_{k:05d}"; k+=1; count=0
            groups.loc[idx]=cur; last=num; count+=1
        for idx in part[part.numeric_index.isna()].index:
            groups.loc[idx]=f"seq_{cls}_{k:05d}"; k+=1
    return groups

component_cache=OUTPUT_DIR/"leakage_component_index.csv"
component_params={"near_duplicate_hamming":CFG.near_duplicate_hamming,"sequence_gap":CFG.sequence_gap,"sequence_block_size":CFG.sequence_block_size}
if component_cache.exists():
    cached=pd.read_csv(component_cache)
    cache_ok = set(cached.relative_path)==set(hash_df.relative_path)
    for key,value in component_params.items():
        cache_ok = cache_ok and key in cached.columns and (cached[key].astype(str)==str(value)).all()
    if cache_ok:
        hash_df=hash_df.merge(cached[["relative_path","phash_group","sequence_group"]], on="relative_path", how="left")
        print("Loaded cached components:", component_cache)
    else:
        print("Ignoring stale component cache; rebuilding components with current parameters.")
if "phash_group" not in hash_df or hash_df.phash_group.isna().any():
    hash_df["phash_group"]=build_perceptual_components(hash_df, CFG.near_duplicate_hamming)
    hash_df["sequence_group"]=build_sequence_proxy_groups(hash_df, CFG.sequence_gap, CFG.sequence_block_size)
    for key,value in component_params.items(): hash_df[key]=value
    hash_df.to_csv(component_cache, index=False)
    print("Saved component cache:", component_cache)

phash_summary=hash_df.groupby("phash_group").agg(n=("relative_path","size"), classes=("class_name",lambda s:", ".join(sorted(set(s)))), examples=("relative_path",lambda s:" | ".join(list(s)[:5]))).reset_index().sort_values("n", ascending=False)
seq_summary=hash_df.groupby("sequence_group").agg(n=("relative_path","size"), classes=("class_name",lambda s:", ".join(sorted(set(s)))), min_index=("numeric_index","min"), max_index=("numeric_index","max"), examples=("relative_path",lambda s:" | ".join(list(s)[:5]))).reset_index().sort_values("n", ascending=False)
print("Perceptual components:", len(phash_summary), "| with >1 image:", int((phash_summary.n>1).sum()))
print("Sequence proxy groups:", len(seq_summary), "| with >1 image:", int((seq_summary.n>1).sum()))
display(phash_summary.head(20)); display(seq_summary.head(20))



## 4) Leakage-Aware Split Builder



In [ ]:
def group_stratified_split(df, group_col, train_frac, val_frac, test_frac, seed):
    rng=random.Random(seed); assign={}
    meta=df.groupby(group_col).agg(n=("relative_path","size"), majority_label=("label",lambda s:Counter(s).most_common(1)[0][0])).reset_index()
    for label,groups in meta.groupby("majority_label"):
        rows=groups.to_dict("records"); rng.shuffle(rows); rows=sorted(rows, key=lambda r:int(r["n"]), reverse=True)
        total=sum(int(r["n"]) for r in rows)
        targets={"train":total*train_frac,"val":total*val_frac,"test":total*test_frac}
        counts={"train":0,"val":0,"test":0}
        for r in rows:
            deficits={s:targets[s]-counts[s] for s in counts}
            split=max(deficits, key=deficits.get) if max(deficits.values())>0 else min(counts, key=counts.get)
            assign[r[group_col]]=split; counts[split]+=int(r["n"])
    return df[group_col].map(assign)

def build_all_splits(df):
    specs={"random":"relative_path","sha256_clean":"sha256","phash_clean":"phash_group","sequence_proxy":"sequence_group"}
    frames=[]
    for mode,gcol in specs.items():
        m=df.copy(); m["split_mode"]=mode; m["group_col"]=gcol; m["group_id"]=m[gcol].astype(str)
        m["split"]=group_stratified_split(m,gcol,CFG.train_split,CFG.val_split,CFG.test_split,CFG.split_seed)
        frames.append(m)
    return pd.concat(frames, ignore_index=True)

def cross_split_group_stats(df, group_col):
    groups=images=pairs=0
    for _,part in df.groupby(group_col):
        counts=part.groupby("split").size().to_dict()
        active={k:v for k,v in counts.items() if v>0}
        if len(active)<=1: continue
        groups+=1; images+=sum(active.values())
        for a,b in combinations(active.values(),2): pairs+=int(a*b)
    return {"groups":groups,"images":images,"pairs":pairs}

def validate_split(mode_df):
    gcol=mode_df.group_col.iloc[0]
    exact=cross_split_group_stats(mode_df,"sha256")
    near=cross_split_group_stats(mode_df,"phash_group")
    declared=cross_split_group_stats(mode_df,gcol)
    counts=mode_df.groupby("split").size().to_dict()
    return dict(split_mode=mode_df.split_mode.iloc[0], declared_group_col=gcol,
        train_n=int(counts.get("train",0)), val_n=int(counts.get("val",0)), test_n=int(counts.get("test",0)),
        declared_group_cross_split_groups=declared["groups"],
        exact_cross_split_duplicate_groups=exact["groups"], exact_cross_split_duplicate_pairs=exact["pairs"],
        near_cross_split_component_groups=near["groups"], near_cross_split_component_pairs=near["pairs"])

split_assignments=build_all_splits(hash_df)
split_csv=OUTPUT_DIR/"split_assignments.csv"
split_assignments[["relative_path","class_name","label","split_mode","split","group_col","group_id","sha256","phash_group","sequence_group"]].to_csv(split_csv, index=False)
print("Saved split assignments:", split_csv)
validation_rows=[]
for mode,mdf in split_assignments.groupby("split_mode"):
    print(f"\n{mode} class distribution:")
    display(mdf.groupby(["split","class_name"]).size().unstack(fill_value=0))
    validation_rows.append(validate_split(mdf))
split_validation_df=pd.DataFrame(validation_rows).sort_values("split_mode")
display(split_validation_df)



## 5) EfficientNet-B4 Baseline



In [ ]:
def round_filters(f,cfg): 
    f*=cfg.width_coefficient; d=cfg.depth_divisor
    nf=max(d,int(f+d/2)//d*d)
    return int(nf+d if nf<0.9*f else nf)
def round_repeats(r,cfg): return int(math.ceil(cfg.depth_coefficient*r))
class Swish(nn.Module):
    def forward(self,x): return x*torch.sigmoid(x)
def drop_connect(x,p,training):
    if not training or p==0: return x
    keep=1-p; mask=(keep+torch.rand((x.shape[0],1,1,1),device=x.device,dtype=x.dtype)).floor()
    return x/keep*mask
class Conv2dSame(nn.Conv2d):
    def __init__(self,ic,oc,kernel_size,stride=1,groups=1,bias=False):
        super().__init__(ic,oc,kernel_size,stride=stride,padding=0,groups=groups,bias=bias)
    def forward(self,x):
        ih,iw=x.shape[-2:]; kh,kw=self.kernel_size; sh,sw=self.stride; dh,dw=self.dilation
        ph=max((math.ceil(ih/sh)-1)*sh+(kh-1)*dh+1-ih,0); pw=max((math.ceil(iw/sw)-1)*sw+(kw-1)*dw+1-iw,0)
        if ph or pw: x=F.pad(x,[pw//2,pw-pw//2,ph//2,ph-ph//2])
        return F.conv2d(x,self.weight,self.bias,self.stride,0,self.dilation,self.groups)
class SqueezeExcitation(nn.Module):
    def __init__(self,ic,sc):
        super().__init__(); self.pool=nn.AdaptiveAvgPool2d(1); self.reduce=nn.Conv2d(ic,sc,1); self.expand=nn.Conv2d(sc,ic,1); self.act=Swish()
    def forward(self,x):
        s=self.expand(self.act(self.reduce(self.pool(x))))
        return x*torch.sigmoid(s)
class ECABlock(nn.Module):
    def __init__(self,c,k=3):
        super().__init__(); self.pool=nn.AdaptiveAvgPool2d(1); self.conv=nn.Conv1d(1,1,kernel_size=k,padding=(k-1)//2,bias=False)
    def forward(self,x):
        y=self.pool(x).squeeze(-1).transpose(-1,-2); y=self.conv(y).transpose(-1,-2).unsqueeze(-1)
        return x*torch.sigmoid(y)
class MBConvBlock(nn.Module):
    def __init__(self,ic,oc,k,s,exp,se_ratio=.25,drop=.0,attention="se"):
        super().__init__(); self.use_res=s==1 and ic==oc; self.drop=drop; ec=ic*exp; sc=max(1,int(ic*se_ratio)); layers=[]
        if exp!=1: layers += [Conv2dSame(ic,ec,1), nn.BatchNorm2d(ec), Swish()]
        layers += [Conv2dSame(ec,ec,k,stride=s,groups=ec), nn.BatchNorm2d(ec), Swish()]
        if attention=="se": layers.append(SqueezeExcitation(ec,sc))
        elif attention=="eca": layers.append(ECABlock(ec))
        layers += [Conv2dSame(ec,oc,1), nn.BatchNorm2d(oc)]
        self.block=nn.Sequential(*layers)
    def forward(self,x):
        y=self.block(x)
        return x+drop_connect(y,self.drop,self.training) if self.use_res else y
BASE_BLOCKS=[(1,16,1,1,3),(6,24,2,2,3),(6,40,2,2,5),(6,80,3,2,3),(6,112,3,1,5),(6,192,4,2,5),(6,320,1,1,3)]
class EfficientNet(nn.Module):
    def __init__(self,cfg):
        super().__init__(); stem=round_filters(32,cfg)
        self.stem=nn.Sequential(Conv2dSame(3,stem,3,stride=2), nn.BatchNorm2d(stem), Swish())
        specs=[(e,round_filters(ch,cfg),round_repeats(r,cfg),s,k) for e,ch,r,s,k in BASE_BLOCKS]
        total=sum(x[2] for x in specs); idx=0; ic=stem; blocks=[]
        for e,oc,repeats,s,k in specs:
            for i in range(repeats):
                blocks.append(MBConvBlock(ic,oc,k,s if i==0 else 1,e,drop=cfg.drop_connect_rate*idx/max(1,total-1),attention=cfg.attention_type))
                ic=oc; idx+=1
        self.blocks=nn.Sequential(*blocks); head=round_filters(1280,cfg)
        self.head=nn.Sequential(Conv2dSame(ic,head,1), nn.BatchNorm2d(head), Swish())
        self.pool=nn.AdaptiveAvgPool2d(1); self.dropout=nn.Dropout(cfg.dropout_rate); self.classifier=nn.Linear(head,cfg.num_classes)
        self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m,nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode="fan_out")
            elif isinstance(m,nn.BatchNorm2d): nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m,nn.Linear): nn.init.normal_(m.weight,0,.01); nn.init.zeros_(m.bias)
    def forward(self,x):
        x=self.head(self.blocks(self.stem(x))); x=torch.flatten(self.pool(x),1)
        return self.classifier(self.dropout(x))
model_audit=EfficientNet(CFG).to(DEVICE)
print("Trainable params:", f"{sum(p.numel() for p in model_audit.parameters() if p.requires_grad):,}")
del model_audit
if DEVICE.type=="cuda": torch.cuda.empty_cache()



## 6) DataLoaders, Training, Metrics, and Calibration



In [ ]:
def build_transforms(cfg):
    train=transforms.Compose([transforms.Resize((cfg.input_resolution,cfg.input_resolution)), transforms.Grayscale(3),
        transforms.RandomHorizontalFlip(.5), transforms.RandomRotation(10), transforms.ToTensor(), transforms.Normalize(cfg.mean,cfg.std)])
    eval_=transforms.Compose([transforms.Resize((cfg.input_resolution,cfg.input_resolution)), transforms.Grayscale(3),
        transforms.ToTensor(), transforms.Normalize(cfg.mean,cfg.std)])
    return train,eval_
class KidneyCTFrameDataset(Dataset):
    def __init__(self, frame, transform=None): self.frame=frame.reset_index(drop=True).copy(); self.transform=transform
    def __len__(self): return len(self.frame)
    def __getitem__(self,i):
        r=self.frame.iloc[i]
        with Image.open(r.path) as im: img=im.convert("RGB")
        if self.transform: img=self.transform(img)
        return img,int(r.label),r.relative_path
def make_loaders(mode_df,cfg):
    train_tfms,eval_tfms=build_transforms(cfg); frames={s:mode_df[mode_df.split==s].copy() for s in ["train","val","test"]}
    loader_kwargs=dict(num_workers=cfg.num_workers, pin_memory=(DEVICE.type=="cuda"))
    # Avoid creating empty loaders; this turns split bugs into clear assertions before training starts.
    for split_name, frame in frames.items():
        assert len(frame)>0, f"{split_name} split is empty for this split mode; inspect split_validation_df before training."
    loaders={"train":DataLoader(KidneyCTFrameDataset(frames["train"],train_tfms),cfg.batch_size,shuffle=True,**loader_kwargs),
             "val":DataLoader(KidneyCTFrameDataset(frames["val"],eval_tfms),cfg.batch_size,shuffle=False,**loader_kwargs),
             "test":DataLoader(KidneyCTFrameDataset(frames["test"],eval_tfms),cfg.batch_size,shuffle=False,**loader_kwargs)}
    return loaders,frames
def train_one_epoch(model,loader,opt,crit,scaler,epoch,total):
    model.train(); loss_sum=correct=n=0
    for x,y,_ in tqdm(loader, desc=f"Epoch {epoch}/{total} [train]", leave=False):
        x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True); opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=(CFG.amp and DEVICE.type=="cuda")):
            logits=model(x); loss=crit(logits,y)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        if CFG.grad_clip_norm: nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip_norm)
        scaler.step(opt); scaler.update()
        bs=x.size(0); loss_sum+=loss.item()*bs; correct+=(logits.argmax(1)==y).sum().item(); n+=bs
    return {"loss":loss_sum/n,"accuracy":correct/n}
@torch.no_grad()
def collect_logits(model,loader,crit):
    model.eval(); L=[]; Y=[]; paths=[]; loss_sum=n=0
    for x,y,p in tqdm(loader, desc="collect logits", leave=False):
        x=x.to(DEVICE,non_blocking=True); yd=y.to(DEVICE,non_blocking=True); logits=model(x); loss=crit(logits,yd)
        bs=x.size(0); loss_sum+=loss.item()*bs; n+=bs; L.append(logits.cpu()); Y.append(y.cpu()); paths+=list(p)
    logits=torch.cat(L); targets=torch.cat(Y); probs=torch.softmax(logits,1)
    return {"loss":loss_sum/n,"logits":logits,"probs":probs,"targets":targets,"paths":paths}
def confusion_matrix_np(y_true,y_pred,k):
    cm=np.zeros((k,k),dtype=np.int64)
    for t,p in zip(y_true,y_pred): cm[int(t),int(p)]+=1
    return cm
def classification_metrics(targets,probs,class_names):
    y=targets.numpy().astype(int); pred=probs.argmax(1).numpy().astype(int); cm=confusion_matrix_np(y,pred,len(class_names))
    rows=[]; ps=[]; rs=[]; fs=[]
    for i,name in enumerate(class_names):
        tp=cm[i,i]; fp=cm[:,i].sum()-tp; fn=cm[i,:].sum()-tp
        p=tp/max(1,tp+fp); r=tp/max(1,tp+fn); f=2*p*r/max(1e-12,p+r)
        ps.append(p); rs.append(r); fs.append(f); rows.append({"class_name":name,"precision":p,"recall":r,"f1":f,"support":int(cm[i,:].sum())})
    auc=np.nan
    if roc_auc_score is not None:
        try: auc=float(roc_auc_score(np.eye(len(class_names))[y], probs.numpy(), multi_class="ovr", average="macro"))
        except Exception: pass
    return {"accuracy":float((y==pred).mean()),"macro_precision":float(np.mean(ps)),"macro_recall":float(np.mean(rs)),
            "macro_f1":float(np.mean(fs)),"macro_auc_ovr":auc,"confusion_matrix":cm,"per_class":pd.DataFrame(rows)}
def brier_score_multiclass(targets,probs,k):
    return float(torch.mean(torch.sum((probs.float()-F.one_hot(targets.long(),k).float())**2,1)).item())
def expected_calibration_error(targets,probs,n_bins=15):
    conf,pred=probs.max(1); acc=pred.eq(targets); ece=torch.zeros(1); bins=torch.linspace(0,1,n_bins+1)
    for lo,hi in zip(bins[:-1],bins[1:]):
        m=conf.gt(lo)&conf.le(hi); prop=m.float().mean()
        if prop.item()>0: ece += torch.abs(conf[m].mean()-acc[m].float().mean())*prop
    return float(ece.item())
def fit_temperature(logits,targets,max_iter=50):
    log_t=torch.zeros(1,requires_grad=True); opt=torch.optim.LBFGS([log_t],lr=.1,max_iter=max_iter); crit=nn.CrossEntropyLoss()
    def closure():
        opt.zero_grad(); t=torch.exp(log_t).clamp(.05,20); loss=crit(logits/t,targets.long()); loss.backward(); return loss
    opt.step(closure); return float(torch.exp(log_t).detach().clamp(.05,20).item())
def plot_reliability(targets,probs,title,n_bins=10):
    conf,pred=probs.max(1); acc=pred.eq(targets).float(); bins=torch.linspace(0,1,n_bins+1); xs=[]; ys=[]
    for lo,hi in zip(bins[:-1],bins[1:]):
        m=conf.gt(lo)&conf.le(hi); xs.append(float((lo+hi)/2 if not m.any() else conf[m].mean())); ys.append(np.nan if not m.any() else float(acc[m].mean()))
    fig,ax=plt.subplots(figsize=(5,5)); ax.plot([0,1],[0,1],"--",color="gray"); ax.plot(xs,ys,marker="o")
    ax.set(xlim=(0,1),ylim=(0,1),xlabel="Confidence",ylabel="Accuracy",title=title); ax.grid(alpha=.25); plt.show()



## 7) Benchmark Runner



In [ ]:
def fit_one_split(mode_df,mode,cfg,epochs):
    loaders,frames=make_loaders(mode_df,cfg); model=EfficientNet(cfg).to(DEVICE)
    crit=nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    opt=torch.optim.AdamW(model.parameters(),lr=cfg.learning_rate,weight_decay=cfg.weight_decay)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs)
    scaler=torch.amp.GradScaler("cuda",enabled=(cfg.amp and DEVICE.type=="cuda"))
    history=[]; best=-1; best_state=None; best_epoch=0; stale=0; start=time.time()
    for epoch in range(1,epochs+1):
        tr=train_one_epoch(model,loaders["train"],opt,crit,scaler,epoch,epochs)
        val=collect_logits(model,loaders["val"],crit); vm=classification_metrics(val["targets"],val["probs"],CLASS_NAMES); sched.step()
        row={"epoch":epoch,"train_loss":tr["loss"],"train_accuracy":tr["accuracy"],"val_loss":val["loss"],"val_accuracy":vm["accuracy"],"val_macro_f1":vm["macro_f1"]}
        history.append(row); print(f"[{mode}] epoch {epoch:02d}/{epochs} train_acc={tr['accuracy']:.4f} val_acc={vm['accuracy']:.4f} val_f1={vm['macro_f1']:.4f}")
        if vm["accuracy"]>best:
            best=vm["accuracy"]; best_epoch=epoch; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; stale=0
        else: stale+=1
        if stale>=cfg.early_stop_patience: print(f"[{mode}] early stop; best epoch {best_epoch}"); break
    if best_state: model.load_state_dict(best_state)
    val=collect_logits(model,loaders["val"],crit); test=collect_logits(model,loaders["test"],crit)
    temp=fit_temperature(val["logits"],val["targets"]); scaled=torch.softmax(test["logits"]/temp,1)
    path=OUTPUT_DIR/f"efficientnet_b4_{mode}_best.pth"
    torch.save({"model_state_dict":model.state_dict(),"cfg":asdict(cfg),"class_names":CLASS_NAMES,"split_mode":mode,"best_epoch":best_epoch,"temperature":temp}, path)
    return {"split_mode":mode,"epochs_completed":len(history),"best_epoch":best_epoch,"runtime_minutes":(time.time()-start)/60,
            "checkpoint_path":str(path),"history":pd.DataFrame(history),"val_eval":val,"test_eval":test,"temperature":temp,
            "test_probs_scaled":scaled,"test_metrics":classification_metrics(test["targets"],test["probs"],CLASS_NAMES),
            "test_metrics_scaled":classification_metrics(test["targets"],scaled,CLASS_NAMES),"split_frames":frames}

if CFG.execution_mode=="audit_only": benchmark_modes=[]; epochs=0
elif CFG.execution_mode=="smoke": benchmark_modes=["random"]; epochs=CFG.smoke_epochs
elif CFG.execution_mode=="full": benchmark_modes=["random","sha256_clean","phash_clean","sequence_proxy"]; epochs=CFG.full_epochs
else: raise ValueError("execution_mode must be audit_only, smoke, or full")
benchmark_results={}
for mode in benchmark_modes:
    print(f"\n=== Running split mode: {mode} ===")
    benchmark_results[mode]=fit_one_split(split_assignments[split_assignments.split_mode==mode].copy(), mode, CFG, epochs)
print("Completed:", list(benchmark_results))



## 8) Final Tables and Calibration Plots



In [ ]:
def summarize_result(mode,result,split_validation_df):
    s=split_validation_df[split_validation_df.split_mode==mode].iloc[0].to_dict()
    test=result["test_eval"]; m=result["test_metrics"]; ms=result["test_metrics_scaled"]; scaled=result["test_probs_scaled"]
    return {"split_mode":mode,"train_n":int(s["train_n"]),"val_n":int(s["val_n"]),"test_n":int(s["test_n"]),
        "exact_cross_split_duplicate_groups":int(s["exact_cross_split_duplicate_groups"]),
        "exact_cross_split_duplicate_pairs":int(s["exact_cross_split_duplicate_pairs"]),
        "near_cross_split_component_groups":int(s["near_cross_split_component_groups"]),
        "near_cross_split_component_pairs":int(s["near_cross_split_component_pairs"]),
        "test_accuracy":m["accuracy"],"test_macro_f1":m["macro_f1"],"test_macro_auc_ovr":m["macro_auc_ovr"],
        "test_brier":brier_score_multiclass(test["targets"],test["probs"],len(CLASS_NAMES)),
        "test_ece":expected_calibration_error(test["targets"],test["probs"]),"temperature":result["temperature"],
        "scaled_test_accuracy":ms["accuracy"],"scaled_test_macro_f1":ms["macro_f1"],
        "scaled_test_brier":brier_score_multiclass(test["targets"],scaled,len(CLASS_NAMES)),
        "scaled_test_ece":expected_calibration_error(test["targets"],scaled),
        "epochs_completed":result["epochs_completed"],"best_epoch":result["best_epoch"]}

if benchmark_results:
    summary_df=pd.DataFrame([summarize_result(m,r,split_validation_df) for m,r in benchmark_results.items()])
    summary_path=OUTPUT_DIR/"benchmark_summary.csv"; summary_df.to_csv(summary_path,index=False); print("Saved:", summary_path)
    display(summary_df)
    for mode,result in benchmark_results.items():
        print(f"\n{mode}: confusion matrix"); display(pd.DataFrame(result["test_metrics"]["confusion_matrix"], index=CLASS_NAMES, columns=CLASS_NAMES))
        print(f"\n{mode}: per-class metrics"); display(result["test_metrics"]["per_class"])
        plot_reliability(result["test_eval"]["targets"], result["test_eval"]["probs"], f"{mode} reliability before scaling")
        plot_reliability(result["test_eval"]["targets"], result["test_probs_scaled"], f"{mode} reliability after scaling")
else:
    print("No model benchmark was run. Set CFG.execution_mode to 'smoke' or 'full'.")



## 9) Grad-CAM XAI Sanity Check



In [ ]:
class GradCAM:
    def __init__(self,model,target_layer):
        self.model=model; self.activations=None; self.gradients=None
        self.handles=[target_layer.register_forward_hook(self.fwd), target_layer.register_full_backward_hook(self.bwd)]
    def fwd(self,_,__,out): self.activations=out.detach()
    def bwd(self,_,__,grad_out): self.gradients=grad_out[0].detach()
    def remove(self):
        for h in self.handles: h.remove()
    def __call__(self,x,class_idx=None):
        self.model.eval(); logits=self.model(x); class_idx=int(logits.argmax(1).item()) if class_idx is None else class_idx
        self.model.zero_grad(set_to_none=True); logits[:,class_idx].sum().backward(retain_graph=True)
        w=self.gradients.mean((2,3),keepdim=True); cam=F.relu((w*self.activations).sum(1,keepdim=True))
        cam=F.interpolate(cam,size=x.shape[-2:],mode="bilinear",align_corners=False).squeeze().detach().cpu()
        return (cam-cam.min())/(cam.max()-cam.min()+1e-8), logits.detach().cpu()
def denormalize(x,cfg):
    return (x.cpu()*torch.tensor(cfg.std).view(3,1,1)+torch.tensor(cfg.mean).view(3,1,1)).clamp(0,1)
def choose_xai_examples(result,max_examples=4):
    e=result["test_eval"]; probs=e["probs"]; targets=e["targets"]; pred=probs.argmax(1); conf=probs.max(1).values
    chosen=[i for i in range(len(e["paths"])) if int(pred[i])!=int(targets[i])][:max_examples]
    for i in torch.argsort(conf).tolist():
        if len(chosen)>=max_examples: break
        if i not in chosen: chosen.append(i)
    return [e["paths"][i] for i in chosen]
def show_gradcam_for_paths(mode,result,cfg,max_examples=4):
    ckpt=torch.load(result["checkpoint_path"],map_location=DEVICE); model=EfficientNet(cfg).to(DEVICE)
    model.load_state_dict(ckpt["model_state_dict"]); model.eval(); cammer=GradCAM(model,model.blocks); _,eval_tfms=build_transforms(cfg)
    rows=result["split_frames"]["test"].set_index("relative_path").loc[choose_xai_examples(result,max_examples)].reset_index()
    fig,axes=plt.subplots(len(rows),3,figsize=(11,3.5*len(rows)))
    if len(rows)==1: axes=np.expand_dims(axes,0)
    try:
        for i,r in rows.iterrows():
            with Image.open(r.path) as im: img=im.convert("RGB")
            x=eval_tfms(img).unsqueeze(0).to(DEVICE); cam,logits=cammer(x); probs=torch.softmax(logits,1)[0]; pred=int(probs.argmax())
            base=denormalize(x[0],cfg).permute(1,2,0).numpy()
            axes[i,0].imshow(base); axes[i,0].set_title(f"True: {CLASS_NAMES[int(r.label)]}")
            axes[i,1].imshow(cam.numpy(),cmap="jet"); axes[i,1].set_title("Grad-CAM")
            axes[i,2].imshow(base); axes[i,2].imshow(cam.numpy(),cmap="jet",alpha=.45); axes[i,2].set_title(f"Pred: {CLASS_NAMES[pred]} ({float(probs[pred]):.3f})")
            for ax in axes[i]: ax.axis("off")
        fig.suptitle(f"{mode} Grad-CAM sanity check", y=1.01); plt.tight_layout(); plt.show()
    finally:
        cammer.remove(); del model
        if DEVICE.type=="cuda": torch.cuda.empty_cache()
for mode,result in benchmark_results.items(): show_gradcam_for_paths(mode,result,CFG,4)



## 10) Conclusion Template



In [ ]:
if benchmark_results:
    cols=["split_mode","exact_cross_split_duplicate_groups","near_cross_split_component_groups","test_accuracy","test_macro_f1","test_ece","scaled_test_ece"]
    display(summary_df[cols])
    print("Conclusion draft:")
    print("Random image-level splitting is expected to be optimistic on this JPEG CT dataset because exact and perceptual near-duplicate images can cross train/validation/test boundaries. The duplicate-disjoint and perceptual-component-disjoint splits should be treated as the credible benchmark results. Because reliable patient/study identifiers are not available in the public JPEG files, future clinical validation should use metadata-backed patient- or study-disjoint evaluation.")
else:
    display(split_validation_df)
    print("Audit complete. For full benchmark evidence, set CFG.execution_mode = 'full' and rerun from the configuration cell.")
